# E-Commerce Sales Analysis - Exploratory Data Analysis

This notebook provides an interactive exploration of the Superstore dataset.

## Contents
1. Data Loading
2. Data Overview
3. Revenue Analysis
4. Regional Performance
5. Product Analysis
6. Customer Segmentation
7. Profit Analysis

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

# Set visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

print("✓ Libraries imported successfully")

## 1. Data Loading

In [ ]:
# Load the cleaned dataset
data_path = Path('../data/processed/superstore_clean.csv')

if data_path.exists():
    df = pd.read_csv(data_path, parse_dates=['Order Date', 'Ship Date'])
    print(f"✓ Data loaded: {len(df)} rows, {len(df.columns)} columns")
else:
    print("❌ Cleaned data not found. Please run: python ../src/data_loader.py")
    df = None

## 2. Data Overview

In [ ]:
# Display first few rows
df.head()

In [ ]:
# Dataset information
df.info()

In [ ]:
# Summary statistics
df.describe()

In [ ]:
# Key metrics
print("="*60)
print("KEY BUSINESS METRICS")
print("="*60)
print(f"Total Sales: ${df['Sales'].sum():,.2f}")
print(f"Total Profit: ${df['Profit'].sum():,.2f}")
print(f"Profit Margin: {(df['Profit'].sum()/df['Sales'].sum()*100):.2f}%")
print(f"Total Orders: {df['Order ID'].nunique():,}")
print(f"Total Customers: {df['Customer ID'].nunique():,}")
print(f"Total Products: {df['Product ID'].nunique():,}")
print(f"Average Order Value: ${df.groupby('Order ID')['Sales'].sum().mean():,.2f}")
print("="*60)

## 3. Revenue Analysis

In [ ]:
# Monthly revenue trend
monthly_sales = df.groupby('Year_Month')['Sales'].sum().reset_index()

fig = px.line(monthly_sales, x='Year_Month', y='Sales',
              title='Monthly Revenue Trend',
              labels={'Sales': 'Revenue ($)', 'Year_Month': 'Month'})
fig.update_traces(line_color='#06A77D', line_width=3)
fig.show()

In [ ]:
# Yearly revenue comparison
yearly_sales = df.groupby('Year')['Sales'].sum().reset_index()

fig = px.bar(yearly_sales, x='Year', y='Sales',
             title='Annual Revenue Comparison',
             labels={'Sales': 'Revenue ($)'},
             text='Sales')
fig.update_traces(texttemplate='$%{text:.2s}', textposition='outside')
fig.show()

## 4. Regional Performance

In [ ]:
# Sales by region
regional_sales = df.groupby('Region')['Sales'].sum().reset_index()

fig = px.pie(regional_sales, values='Sales', names='Region',
             title='Sales Distribution by Region',
             color_discrete_sequence=px.colors.qualitative.Set2)
fig.update_traces(textposition='inside', textinfo='percent+label')
fig.show()

In [ ]:
# Top 10 states
top_states = df.groupby('State')['Sales'].sum().nlargest(10).reset_index()

fig = px.bar(top_states, x='Sales', y='State', orientation='h',
             title='Top 10 States by Sales',
             labels={'Sales': 'Revenue ($)'},
             color='Sales',
             color_continuous_scale='Blues')
fig.show()

## 5. Product Analysis

In [ ]:
# Category performance
category_perf = df.groupby('Category').agg({
    'Sales': 'sum',
    'Profit': 'sum',
    'Order ID': 'count'
}).round(2)

category_perf['Profit_Margin_%'] = (category_perf['Profit'] / category_perf['Sales'] * 100).round(2)
category_perf

In [ ]:
# Top 10 products by sales
top_products = df.groupby('Product Name')['Sales'].sum().nlargest(10).reset_index()

fig = px.bar(top_products, x='Sales', y='Product Name', orientation='h',
             title='Top 10 Products by Sales',
             labels={'Sales': 'Revenue ($)'},
             color='Sales',
             color_continuous_scale='Viridis')
fig.show()

## 6. Customer Segmentation

In [ ]:
# Segment analysis
segment_analysis = df.groupby('Segment').agg({
    'Sales': 'sum',
    'Profit': 'sum',
    'Customer ID': 'nunique',
    'Order ID': 'count'
}).round(2)

segment_analysis.columns = ['Total Sales', 'Total Profit', 'Customers', 'Orders']
segment_analysis['Avg Order Value'] = (segment_analysis['Total Sales'] / segment_analysis['Orders']).round(2)
segment_analysis

In [ ]:
# Segment by category
segment_category = pd.crosstab(df['Segment'], df['Category'], 
                               values=df['Sales'], aggfunc='sum')

fig = px.imshow(segment_category,
                labels=dict(x="Category", y="Segment", color="Sales ($)"),
                title='Sales Heatmap: Segment vs Category',
                color_continuous_scale='YlOrRd',
                text_auto='.0f')
fig.show()

## 7. Profit Analysis

In [ ]:
# Profit margin by category
profit_by_category = df.groupby('Category').agg({
    'Sales': 'sum',
    'Profit': 'sum'
}).reset_index()
profit_by_category['Profit_Margin_%'] = (profit_by_category['Profit'] / 
                                          profit_by_category['Sales'] * 100).round(2)

fig = px.bar(profit_by_category, x='Category', y='Profit_Margin_%',
             title='Profit Margin by Category',
             labels={'Profit_Margin_%': 'Profit Margin (%)'},
             color='Profit_Margin_%',
             color_continuous_scale='RdYlGn')
fig.show()

In [ ]:
# Discount impact
df['Discount_Range'] = pd.cut(df['Discount'], 
                              bins=[-0.01, 0, 0.1, 0.2, 0.3, 1.0],
                              labels=['No Discount', '1-10%', '11-20%', '21-30%', '>30%'])

discount_impact = df.groupby('Discount_Range').agg({
    'Profit_Margin': 'mean',
    'Sales': 'sum',
    'Order ID': 'count'
}).round(2)

discount_impact.columns = ['Avg Profit Margin %', 'Total Sales', 'Orders']
discount_impact

## Custom Analysis

Add your own analysis cells below:

In [ ]:
# Your custom analysis here